# 📦 ASOS Inventory & Phantom Revenue Analysis
## Data Analytics Portfolio Project

### **Business Problem**
ASOS, a leading online fashion retailer, experiences stockouts on popular items. Each out-of-stock size represents **lost revenue** (phantom revenue). This analysis quantifies these losses and identifies which brands and products are most affected, providing actionable recommendations to optimize inventory.

**Dataset:** Scraped product data from ASOS (CSV file).  
**Tools:** Python (Pandas, Matplotlib, Seaborn) in Jupyter Notebook.

In [ ]:
# Import libraries and load data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the CSV file (adjust the file path to your local location)
file_path = r'E:\VS Code\data-analyst-portfolio\Retail_Inventory_Optimization\products_asos.csv'
df = pd.read_csv(file_path, on_bad_lines='skip')

print(f"Raw data loaded: {len(df)} rows")
df.head()

## 1. Data Cleaning – Price Column
The `price` column may contain non‑numeric values (e.g., currency symbols, text). We convert it to numeric and drop rows where conversion fails. This ensures reliable calculations later.

In [ ]:
# Clean price column
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

print(f"After cleaning price: {len(df)} rows remain")
df.head()

## 2. Extract Brand from Product Description
The dataset does **not** have a dedicated brand column. However, the `description` field often contains the brand name after the word **"by"** (e.g., "... by New Look").  
We create a function to extract that word and apply it to all rows.

In [ ]:
# Function to extract brand from description
df['description'] = df['description'].astype(str)

def get_brand(text):
    if 'by' in text:
        try:
            # Split on 'by ', take second part, then split on space and take first word
            return text.split('by ')[1].split(' ')[0]
        except:
            return 'Unknown'
    return 'Unknown'

df['brand_raw'] = df['description'].apply(get_brand)
df[['description', 'brand_raw']].head(10)

## 3. Standardize Brand Names
The extracted raw brand names need cleaning (e.g., `'New'` → `'New Look'` or `'River'` → `'River Island'`). We use a mapping dictionary and then filter out brands with fewer than 5 products to focus on significant players.

In [ ]:
# Map raw brands to proper names and filter
brand_map = {
    'New': 'New Look',
    'River': 'River Island',
    'Bershka': 'Bershka',
    'Miss': 'Miss Selfridge',
    'TopshopWelcome': 'Topshop'
}

df['brand'] = df['brand_raw'].map(brand_map).fillna(df['brand_raw'])

# Keep brands that appear at least 5 times
brand_counts = df['brand'].value_counts()
valid_brands = brand_counts[brand_counts >= 5].index
df_filtered = df[df['brand'].isin(valid_brands)].copy()

print("Top brands after filtering:")
print(df_filtered['brand'].value_counts().head(10))

## 4. Stockout Analysis – Phantom Revenue
The `size` column lists available sizes and may include the text **"Out of stock"** for unavailable ones.  
We create a function that:
- Splits the size string by commas
- Counts how many times `"Out of stock"` appears
- Calculates the stockout rate (out‑of‑stock sizes / total sizes)
- Returns both the count and the rate

Then we apply it to every product and compute **lost revenue** = `price` × `stockout_count`.

In [ ]:
# Define and apply stockout function
def calculate_phantom_revenue(size_str):
    if not isinstance(size_str, str):
        return 0, 0.0
    
    sizes = size_str.split(',')
    total_sizes = len(sizes)
    out_of_stock_count = size_str.count('Out of stock')   # Count how many times 'Out of stock' appears
    rate = out_of_stock_count / total_sizes if total_sizes > 0 else 0.0
    return out_of_stock_count, rate

# Apply function
metrics = df_filtered['size'].apply(lambda x: calculate_phantom_revenue(x))
df_filtered['stockout_count'] = [x[0] for x in metrics]
df_filtered['stockout_rate'] = [x[1] for x in metrics]
df_filtered['lost_revenue'] = df_filtered['price'] * df_filtered['stockout_count']

# Show top 10 products by lost revenue
cols = ['brand', 'name', 'price', 'stockout_count', 'lost_revenue']
print("Top 10 products by lost revenue:")
print(df_filtered.sort_values(by='lost_revenue', ascending=False).head(10)[cols])

## 5. Brand‑Level Aggregation
To get strategic insights, we group by brand and calculate:
- Average price
- Average stockout rate
- Total lost revenue
- Product count (volume)

We keep only brands with at least 10 products.

In [ ]:
# Aggregate by brand
brand_strategy = df_filtered.groupby('brand').agg({
    'price': 'mean',
    'stockout_rate': 'mean',
    'lost_revenue': 'sum',
    'name': 'count'
}).reset_index()

brand_strategy = brand_strategy[brand_strategy['name'] >= 10]
brand_strategy.sort_values('lost_revenue', ascending=False).head(10)

## 6. Visualisation – Brand Strategy Quadrant
We plot each brand as a bubble:
- **X‑axis:** average price  
- **Y‑axis:** average stockout rate  
- **Bubble size:** total lost revenue  

Red dashed lines at `price = 40` and `stockout_rate = 0.4` divide the plot into four strategic quadrants. Brands in the **top‑right quadrant** (high price, high stockout rate) are the **"winners"** – they are expensive AND constantly selling out, representing the biggest missed opportunity.

In [ ]:
# Create scatter plot with quadrant lines
plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=brand_strategy,
    x='price',
    y='stockout_rate',
    size='lost_revenue',
    sizes=(50, 500),
    alpha=0.7,
    palette='viridis',
    legend=False
)

# Identify and label brands in the top‑right quadrant
winners = brand_strategy[
    (brand_strategy['price'] > 40) &
    (brand_strategy['stockout_rate'] > 0.4)
]

for i in range(len(winners)):
    plt.text(
        winners['price'].iloc[i] + 1,
        winners['stockout_rate'].iloc[i],
        winners['brand'].iloc[i],
        fontsize=10,
        ha='left',
        va='bottom'
    )

# Quadrant lines
plt.axvline(x=40, color='red', linestyle='--', linewidth=1)
plt.axhline(y=0.4, color='red', linestyle='--', linewidth=1)

plt.title('Brand Strategy Analysis: Price vs Stockout Rate', fontsize=16)
plt.xlabel('Average Price (£)', fontsize=12)
plt.ylabel('Average Stockout Rate', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Key Insights & Business Recommendations

### **What the chart tells us:**
- **Top‑right quadrant (High Price, High Stockout):** Brands like *Mango*, *Pull&Bear*, *Topshop* are premium and in high demand. They sell out faster than they can be restocked, causing substantial lost revenue.  
  **Recommendation:** Increase inventory levels for these brands immediately – they are the “gold mine”.

- **Top‑left quadrant (Low Price, High Stockout):** Essential basics (e.g., plain t‑shirts, leggings) that fly off the shelves.  
  **Recommendation:** Ensure deep stock of these staples; they drive traffic and repeat purchases.

- **Bottom‑right quadrant (High Price, Low Stockout):** Luxury or niche items that sit on shelves.  
  **Recommendation:** Consider markdowns or bundling to clear inventory; reallocate budget to faster‑moving items.

- **Bottom‑left quadrant (Low Price, Low Stockout):** Safe but low‑growth products.  
  **Recommendation:** Maintain minimal stock; not a priority for investment.

### **Quantified Opportunity**
The total lost revenue across all analysed products is **£{sum_lost:,.0f}** (calculate in a separate cell if desired). By focusing on the top‑right quadrant brands, ASOS could capture a significant portion of this phantom revenue.

### **Next Steps**
- Drill down into individual products within the winning brands to identify exact sizes that cause the most stockouts.
- Work with suppliers to shorten lead times for high‑demand items.
- Use dynamic pricing or pre‑orders for products with chronic stockouts.

---

📌 *This analysis demonstrates how Python can turn raw e‑commerce data into strategic business recommendations. All code and insights are reproducible with the provided dataset.*